# Data Mining Lab Assignment 2
## Recommender System using a Medallion Data Lake

**Name:** Rishabh Kushwaha  
**Roll No.:** LIT2023041

**Objective:** Process the Book-Crossing dataset through Bronze, Silver and Gold layers, then build an item-based collaborative filtering recommender using cosine similarity.

**Data flow:** Raw CSV → Bronze Parquet → EDA → Silver cleaned Parquet → Gold sparse matrix and lookup tables → Book recommendations.


## Step 1 — Set up the data lake and build Bronze
The three source CSV files are retained in the downloaded ZIP. CSV parsing uses semicolon delimiters and ISO-8859-1 (Latin-1) encoding. All source fields are preserved as strings in Bronze, including blank fields and literal `NULL` values. No filtering, normalization or deduplication is performed. A common UTC ingestion timestamp is added.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict
from functools import lru_cache
import csv
import io
import zipfile
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix, save_npz, load_npz
from sklearn.neighbors import NearestNeighbors
from IPython.display import display, Markdown

ROOT = Path('.')
for layer in ('bronze', 'silver', 'gold'):
    (ROOT / layer).mkdir(exist_ok=True)

URL = 'https://cdn.freecodecamp.org/project-data/books/book-crossings.zip'
archive = ROOT / 'book-crossings.zip'
if not archive.exists():
    response = requests.get(URL, timeout=120)
    response.raise_for_status()
    archive.write_bytes(response.content)

print('Data lake folders created.')

In [ ]:
# Resolve filenames by their suffix; the ZIP may contain nested folders.
expected = {'users': 'users.csv', 'books': 'books.csv', 'ratings': 'book-ratings.csv'}
stamp = pd.Timestamp.now(tz='UTC')

def read_source(z, member):
    # Preserve original strings; handle escaped quotes if a record needs it.
    text = z.read(member).decode('latin-1')
    stream = io.StringIO(text, newline='')
    reader = csv.reader(stream, delimiter=';', quotechar='"')
    header = next(reader)
    rows, malformed = [], []
    # The Book-Crossing files are line-oriented. Reparse individual records
    # to avoid a malformed quote swallowing subsequent rows.
    for line_no, raw in enumerate(stream, start=2):
        if not raw.strip():
            continue
        row = next(csv.reader([raw], delimiter=';', quotechar='"'))
        if len(row) != len(header):
            alternative = next(csv.reader([raw], delimiter=';', quotechar='"', escapechar='\\'))
            if len(alternative) == len(header):
                row = alternative
            else:
                malformed.append({'source_line': line_no, 'raw_record': raw.rstrip('\r\n')})
                continue
        rows.append(row)
    # Never silently discard unparseable data: keep exact raw records in Bronze.
    return pd.DataFrame(rows, columns=header, dtype='string'), malformed

with zipfile.ZipFile(archive) as z:
    for table, suffix in expected.items():
        matches = [x for x in z.namelist() if Path(x).name.lower().endswith(suffix)]
        if len(matches) != 1:
            raise ValueError(f'Expected one file ending in {suffix}; found {matches}')
        frame, malformed = read_source(z, matches[0])
        frame['ingestion_timestamp'] = stamp
        frame.to_parquet(ROOT / 'bronze' / f'{table}.parquet', index=False)
        audit = pd.DataFrame(malformed, columns=['source_line', 'raw_record'])
        audit['ingestion_timestamp'] = stamp
        audit.to_parquet(ROOT / 'bronze' / f'{table}_unparsed.parquet', index=False)
        print(f'{table}: {len(frame):,} parsed records; {len(audit):,} unparsed records retained in audit.')

## Step 2 — Exploratory data analysis (EDA)
Read Bronze back from Parquet. For EDA only, temporary numeric views interpret source null tokens as missing values; Bronze remains unchanged.

In [ ]:
bronze = {t: pd.read_parquet(ROOT / 'bronze' / f'{t}.parquet') for t in expected}
u, b, r = (bronze[t] for t in ('users', 'books', 'ratings'))

def missing_view(s):
    s = s.astype('string')
    return s.mask(s.str.strip().str.lower().isin(['', 'null', 'nan', 'n/a']))

def numeric(s):
    return pd.to_numeric(missing_view(s), errors='coerce')

missing_counts = {}
for table, df in bronze.items():
    missing = df.drop(columns='ingestion_timestamp').apply(lambda s: missing_view(s).isna().sum())
    missing_counts[table] = int(missing.sum())
    print(f'\n{table.upper()} — shape: {df.shape}')
    display(pd.DataFrame({'dtype': df.dtypes.astype(str), 'missing': missing.reindex(df.columns, fill_value=0)}))

rating = numeric(r['Book-Rating'])
age = numeric(u['Age'])
year = numeric(b['Year-Of-Publication'])
current_year = datetime.now(timezone.utc).year
print('Rating distribution:')
display(rating.value_counts(dropna=False).sort_index().rename('count').to_frame())
print('Age and year statistics:')
display(pd.DataFrame({'age': age.describe(), 'publication_year': year.describe()}))

user_counts = r.groupby('User-ID').size()
book_counts = r.groupby('ISBN').size()
explicit = r.loc[rating.between(1, 10).fillna(False)]
explicit_user_counts = explicit.groupby('User-ID').size()
explicit_book_counts = explicit.groupby('ISBN').size()
display(pd.DataFrame({'ratings_per_user': user_counts.describe(), 'ratings_per_book': book_counts.describe(),
                      'explicit_per_user': explicit_user_counts.describe(), 'explicit_per_book': explicit_book_counts.describe()}))

# Density uses distinct valid user-book pairs, including recorded zero ratings.
known = r[r['User-ID'].isin(u['User-ID']) & r['ISBN'].isin(b['ISBN'])]
pairs = known[['User-ID', 'ISBN']].drop_duplicates().shape[0]
possible = u['User-ID'].nunique() * b['ISBN'].nunique()
density = 100 * pairs / possible if possible else 0.0
print(f'Observed pair density: {density:.6f}% | Empty pairs (sparsity): {100-density:.6f}%')

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(15, 8))
rating.value_counts().sort_index().plot.bar(ax=ax[0,0], title='Raw rating distribution')
age.dropna().astype(float).plot.hist(bins=40, ax=ax[0,1], title='Raw user ages')
year.dropna().astype(float).plot.hist(bins=40, ax=ax[0,2], title='Raw publication years')
user_counts.value_counts().sort_index().plot(ax=ax[1,0], loglog=True, title='Ratings per user')
book_counts.value_counts().sort_index().plot(ax=ax[1,1], loglog=True, title='Ratings per book')
for a in (ax[1,0], ax[1,1]):
    a.set_xlabel('Number of ratings'); a.set_ylabel('Frequency')
ax[1,2].bar(['Observed', 'Empty'], [density, 100-density]); ax[1,2].set_title('Matrix density (%)')
plt.tight_layout(); plt.show()

### EDA findings and decisions for Silver
- **Shape, types and missing values:** Bronze retains source strings; convert only the necessary numeric fields and handle missing ages and years in Silver.
- **Rating distribution:** Zero represents no explicit rating; retain only ratings from 1 to 10 for modeling.
- **Age and publication year:** Replace impossible ages with missing values and then median-impute; replace zero and future publication years with missing values.
- **Ratings per user/book:** The long tail of interactions motivates a minimum activity threshold of five explicit ratings, less restrictive than ten to retain useful overlaps in sparse data.
- **Matrix sparsity:** A sparse matrix is preferable to allocating a dense matrix for all possible user-book pairs.

The next cell reports the actual numerical findings as rendered Markdown, using the downloaded data rather than hardcoded figures.

In [ ]:
zero_count = int(rating.eq(0).sum())
bad_ages = int(((age < 5) | (age > 100)).sum())
bad_years = int(((year <= 0) | (year > current_year)).sum())
MIN_RATINGS = 5
user_below = 100 * explicit_user_counts.lt(MIN_RATINGS).mean()
book_below = 100 * explicit_book_counts.lt(MIN_RATINGS).mean()
summary = [
    f"**Shape/types/missing:** {', '.join(f'{t}: {len(df):,} rows, {len(df.columns)} columns, {missing_counts[t]:,} missing source values' for t, df in bronze.items())}; convert age/year/rating in Silver.",
    f'**Ratings:** {zero_count:,} zero ratings ({100*zero_count/len(r):.2f}% of source rating rows); remove them from explicit ratings.',
    f'**Ages/years:** {bad_ages:,} out-of-range ages and {bad_years:,} nonpositive or future publication years; null invalid values and median-fill ages.',
    f'**Activity:** median raw ratings per user = {user_counts.median():.0f}, per book = {book_counts.median():.0f}; {user_below:.2f}% of explicitly active users and {book_below:.2f}% of explicitly rated books have fewer than five ratings. Use a five-rating threshold to reduce one-off interactions while retaining overlap.',
    f'**Sparsity:** {pairs:,} observed pairs of {possible:,} possible pairs ({density:.6f}% density); {100-density:.6f}% empty. Use CSR storage.'
]
display(Markdown('\n'.join('- '+s for s in summary)))

## Step 3 — Clean data and build Silver
A minimum of five **distinct explicit** ratings per user and book is used. Filtering is repeated until both conditions hold simultaneously. Conflicting ratings for the same user/book are averaged so they contribute one matrix entry. The auxiliary seen-pair table conservatively records all known interactions (including zero) so they are not recommended again. Zero ratings are never used to train the model.

In [ ]:
users = u.drop(columns='ingestion_timestamp').drop_duplicates().copy()
books = b.drop(columns='ingestion_timestamp').drop_duplicates().copy()
ratings = r.drop(columns='ingestion_timestamp').drop_duplicates().copy()

def clean_text(s):
    return missing_view(s).str.strip().str.replace(r'\s+', ' ', regex=True)

# User IDs and ISBNs remain strings; this preserves leading zeros in ISBNs.
for df, keys in [(users, ['User-ID']), (books, ['ISBN']), (ratings, ['User-ID','ISBN'])]:
    for key in keys:
        df[key] = clean_text(df[key])
    df.dropna(subset=keys, inplace=True)

users['Age'] = numeric(users['Age']).astype('Float64')
users['Age'] = users['Age'].where(users['Age'].between(5, 100))
median_age = users['Age'].median()
if pd.isna(median_age):
    raise ValueError('Cannot impute age: no valid ages in source.')
users['Age'] = users['Age'].fillna(median_age)

# Split from the right so a comma inside the city does not shift the country.
def location_parts(value):
    if pd.isna(value):
        return (pd.NA, pd.NA, pd.NA)
    parts = value.rsplit(',', 2)
    parts = [pd.NA] * (3-len(parts)) + parts
    return tuple(p.strip().lower() if isinstance(p,str) and p.strip() else pd.NA for p in parts)
users['Location'] = clean_text(users['Location'])
users[['city','state','country']] = pd.DataFrame(users['Location'].map(location_parts).tolist(), index=users.index)
users = users.drop(columns='Location').drop_duplicates(subset='User-ID')

books['Year-Of-Publication'] = numeric(books['Year-Of-Publication'])
books['Year-Of-Publication'] = books['Year-Of-Publication'].where(books['Year-Of-Publication'].between(1, current_year))
for col in ['Book-Title', 'Book-Author', 'Publisher']:
    books[col] = clean_text(books[col]).str.lower()
books = books.drop(columns=[col for col in books if col.lower().startswith('image-url')])
books = books.drop_duplicates(subset='ISBN')

ratings['Book-Rating'] = numeric(ratings['Book-Rating'])
ratings = ratings[ratings['User-ID'].isin(users['User-ID']) & ratings['ISBN'].isin(books['ISBN'])].copy()
seen = ratings[['User-ID','ISBN']].drop_duplicates()
ratings = ratings[ratings['Book-Rating'].between(1, 10)].copy()
ratings = ratings.groupby(['User-ID','ISBN'], as_index=False)['Book-Rating'].mean()
print(f'Median age: {median_age}; valid explicit pairs before activity filtering: {len(ratings):,}')

In [ ]:
# Iteratively enforce the activity threshold on both sides of the matrix.
iteration = 0
while True:
    before = len(ratings)
    good_users = ratings['User-ID'].value_counts().loc[lambda s: s >= MIN_RATINGS].index
    good_books = ratings['ISBN'].value_counts().loc[lambda s: s >= MIN_RATINGS].index
    ratings = ratings[ratings['User-ID'].isin(good_users) & ratings['ISBN'].isin(good_books)].copy()
    iteration += 1
    if len(ratings) == before:
        break
if ratings.empty:
    raise ValueError('No ratings survived the threshold. Review EDA and choose a smaller justified threshold.')
users = users[users['User-ID'].isin(ratings['User-ID'])].copy()
books = books[books['ISBN'].isin(ratings['ISBN'])].copy()
seen = seen[seen['User-ID'].isin(users['User-ID']) & seen['ISBN'].isin(books['ISBN'])].copy()

for name, frame in [('users', users), ('books', books), ('ratings', ratings), ('seen_pairs', seen)]:
    frame.to_parquet(ROOT / 'silver' / f'{name}.parquet', index=False)
    print(f'Silver {name}: {frame.shape}')

assert ratings['Book-Rating'].between(1,10).all()
assert not ratings.duplicated(['User-ID','ISBN']).any()
assert ratings.groupby('User-ID').size().min() >= MIN_RATINGS
assert ratings.groupby('ISBN').size().min() >= MIN_RATINGS
assert ratings['User-ID'].isin(users['User-ID']).all()
assert ratings['ISBN'].isin(books['ISBN']).all()
print(f'Silver checks passed after {iteration} filtering iterations.')

## Step 4 — Build Gold
Store a user-by-book sparse CSR matrix with explicit ratings; missing entries are implicit zeros. Store the user and book index mappings, a readable book lookup, and user/book aggregate tables. These mappings are essential for converting matrix indices back to IDs.

In [ ]:
sr = pd.read_parquet(ROOT / 'silver' / 'ratings.parquet')
sb = pd.read_parquet(ROOT / 'silver' / 'books.parquet')
seen_silver = pd.read_parquet(ROOT / 'silver' / 'seen_pairs.parquet')
user_ids = sorted(sr['User-ID'].unique())
book_ids = sorted(sr['ISBN'].unique())
user_map = {uid:i for i,uid in enumerate(user_ids)}
book_map = {isbn:i for i,isbn in enumerate(book_ids)}
rows = sr['User-ID'].map(user_map).to_numpy(dtype=int)
cols = sr['ISBN'].map(book_map).to_numpy(dtype=int)
values = sr['Book-Rating'].to_numpy(dtype=np.float32)
user_book = csr_matrix((values, (rows, cols)), shape=(len(user_ids),len(book_ids)))
save_npz(ROOT / 'gold' / 'user_book_matrix.npz', user_book)

user_index = pd.DataFrame({'user_index': range(len(user_ids)), 'User-ID': user_ids})
book_lookup = pd.DataFrame({'book_index': range(len(book_ids)), 'ISBN': book_ids}).merge(
    sb[['ISBN','Book-Title','Book-Author']], on='ISBN', how='left', validate='one_to_one')
user_index.to_parquet(ROOT / 'gold' / 'user_index.parquet', index=False)
book_lookup.to_parquet(ROOT / 'gold' / 'book_lookup.parquet', index=False)
for key, name in [('User-ID','user_aggregates'), ('ISBN','book_aggregates')]:
    agg = sr.groupby(key)['Book-Rating'].agg(average_rating='mean', rating_count='size').reset_index()
    agg.to_parquet(ROOT / 'gold' / f'{name}.parquet', index=False)
    display(agg.head())
seen_silver.to_parquet(ROOT / 'gold' / 'seen_pairs.parquet', index=False)
assert user_book.nnz == len(sr)
print('Gold matrix (users × books):', user_book.shape)
print('Nonzero ratings:', user_book.nnz)
print(f'Gold matrix density: {100*user_book.nnz/(user_book.shape[0]*user_book.shape[1]):.6f}%')
display(book_lookup.head())

## Step 5 — Item-based collaborative filtering
Transpose Gold to books × users. Cosine similarity compares two books by their vectors of user ratings: cosine(a,b) = (a·b) / (||a|| ||b||). NearestNeighbors computes this on the sparse matrix without storing a dense book-by-book similarity matrix.

For personalized recommendations, ratings of 8–10 are considered high. For each highly rated seed book, collect up to 50 positive-similarity neighbors. Candidate score is the sum of similarity × seed rating divided by the sum of all high seed ratings. Remove previously interacted books; rank by score. If fewer than n candidates have positive similarity, return fewer rather than inventing recommendations.

In [ ]:
# Reload model-ready data independently of the Silver variables.
user_book = load_npz(ROOT / 'gold' / 'user_book_matrix.npz').tocsr()
item_user = user_book.T.tocsr()
lookup = pd.read_parquet(ROOT / 'gold' / 'book_lookup.parquet').sort_values('book_index').reset_index(drop=True)
user_index = pd.read_parquet(ROOT / 'gold' / 'user_index.parquet')
seen_gold = pd.read_parquet(ROOT / 'gold' / 'seen_pairs.parquet')
user_map = dict(zip(user_index['User-ID'].astype(str), user_index['user_index']))
book_map = dict(zip(lookup['ISBN'], lookup['book_index']))
seen_by_user = {str(uid): set(g['ISBN'].map(book_map).dropna().astype(int)) for uid,g in seen_gold.groupby('User-ID')}

model = NearestNeighbors(metric='cosine', algorithm='brute', n_jobs=-1)
model.fit(item_user)
HIGH_RATING = 8
CANDIDATES_PER_SEED = 50

@lru_cache(maxsize=8192)
def neighbors(book_index, count):
    k = min(count+1, item_user.shape[0])
    distances, indices = model.kneighbors(item_user[book_index], n_neighbors=k)
    pairs = [(int(i), float(np.clip(1-d,0,1))) for d,i in zip(distances[0],indices[0]) if int(i)!=book_index]
    return sorted(pairs, key=lambda p: (-p[1], p[0]))[:count]

def book_title(i):
    title = lookup.at[i,'Book-Title']
    return str(title) if pd.notna(title) else f'Untitled (ISBN {lookup.at[i,"ISBN"]})'

def similar_books(title, n=10):
    query = ' '.join(str(title).strip().lower().split())
    matches = lookup.index[lookup['Book-Title'].fillna('').eq(query)].tolist()
    if not matches:
        raise ValueError(f'Book title not found: {title}')
    # Multiple editions can have the same title. Pick the first ISBN in sorted order.
    seed = matches[0]
    return pd.DataFrame([{'ISBN': lookup.at[i,'ISBN'], 'title': book_title(i),
                          'author': lookup.at[i,'Book-Author'], 'similarity': score}
                         for i,score in neighbors(seed,n)], columns=['ISBN','title','author','similarity'])

def highly_rated(user_id):
    uid = str(user_id).strip()
    if uid not in user_map:
        raise ValueError(f'Unknown user ID: {uid}')
    row = user_book.getrow(int(user_map[uid]))
    return sorted([(int(i),float(v)) for i,v in zip(row.indices,row.data) if v>=HIGH_RATING], key=lambda p:(-p[1],p[0]))

def recommend_details(user_id, n=10):
    uid = str(user_id).strip()
    seeds = highly_rated(uid)
    if not seeds or n <= 0:
        return pd.DataFrame(columns=['ISBN','title','author','score'])
    rated = set(user_book.getrow(int(user_map[uid])).indices)
    seen_indices = rated | seen_by_user.get(uid,set())
    seen_titles = {book_title(i) for i in seen_indices}
    scores = defaultdict(float)
    total_weight = sum(v for _,v in seeds)
    for seed, rating in seeds:
        for i, sim in neighbors(seed,CANDIDATES_PER_SEED):
            if sim > 0 and i not in seen_indices and book_title(i) not in seen_titles:
                scores[i] += sim*rating/total_weight
    results, used_titles = [], set()
    for i,score in sorted(scores.items(), key=lambda p:(-p[1],p[0])):
        title = book_title(i)
        if title in used_titles:
            continue
        used_titles.add(title)
        results.append({'ISBN':lookup.at[i,'ISBN'], 'title':title,'author':lookup.at[i,'Book-Author'],'score':score})
        if len(results)>=n:
            break
    return pd.DataFrame(results,columns=['ISBN','title','author','score'])

def recommend(user_id, n=10):
    return recommend_details(user_id,n)['title'].tolist()

example_title = lookup['Book-Title'].dropna().iloc[0]
print('Similar books for:', example_title)
display(similar_books(example_title,10))

## Step 6 — Demo and sanity check
Select three different users with highly rated seed books and available recommendations. Display their highest-rated books alongside their recommended titles. Check for duplicates, invalid scores and previously seen items. These are qualitative sanity checks, not a held-out accuracy evaluation.

In [ ]:
# Select users with many high-rated books to make the demonstration informative.
high_counts = np.asarray((user_book >= HIGH_RATING).sum(axis=1)).ravel()
ordered_users = user_index.sort_values('user_index')['User-ID'].astype(str).tolist()
demo_count = 0
for row_id in np.argsort(-high_counts, kind='stable'):
    if high_counts[row_id] == 0:
        break
    uid = ordered_users[row_id]
    result = recommend_details(uid,10)
    if result.empty:
        continue
    seeds = highly_rated(uid)
    left = pd.DataFrame([{'Highly rated book':book_title(i),'Rating':v} for i,v in seeds])
    right = result[['title','score']].rename(columns={'title':'Recommended book','score':'Similarity score'}).reset_index(drop=True)
    display(Markdown(f'### User ID: {uid}'))
    display(pd.concat([left.reset_index(drop=True),right],axis=1).fillna(''))
    assert len(recommend(uid)) <= 10
    assert not result['title'].duplicated().any()
    assert set(result['ISBN'].map(book_map)).isdisjoint(seen_by_user.get(uid,set()))
    assert result['score'].between(0,1+1e-6).all()
    assert recommend(uid) == result['title'].tolist()
    demo_count += 1
    if demo_count == 3:
        break
assert demo_count == 3, 'Fewer than three users with usable recommendations; review EDA and threshold.'
print('Sanity checks passed for all three demo users.')